# 03 — Evaluate the frozen wallet models

This notebook evaluates the already saved models on the **same original October 4–5 test rows**. It never tunes model settings, chooses a history window, changes the threshold or removes predictors based on test results.

The original project has already inspected these dates. Treat every comparison here as exploratory reuse, not new independent confirmation. The unexecuted delivered notebooks do not contain invented wallet results.

In [ ]:
# Locate this add-on folder whether Jupyter starts here or in 02_notebooks.
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image

here = Path.cwd().resolve()
helper_dir = next((p for base in [here, *here.parents]
                   for p in [base, base / '02_notebooks']
                   if (p / 'rf_common.py').is_file()), None)
if helper_dir is None:
    raise FileNotFoundError('Keep rf_common.py beside these notebooks and start Jupyter in the extracted add-on.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
PACKAGE_ROOT = helper_dir.parent
import rf_common as rf
print('Add-on folder:', PACKAGE_ROOT)

In [ ]:
# Use the exact run created in notebook 01; no original results are overwritten.
OUTPUT_BASE = Path(os.environ.get('WALLET_RF_OUTPUT_BASE', str(PACKAGE_ROOT / '04_results' / 'random_forest'))).expanduser().resolve()
override = os.environ.get('WALLET_RF_RUN_DIR')
RUN_DIR = Path(override).expanduser().resolve() if override else Path(json.loads((OUTPUT_BASE / 'latest_run.json').read_text())['run_dir'])
CONFIG = json.loads((RUN_DIR / 'run_config.json').read_text())
print('Run:', RUN_DIR)
print('Evaluation status:', CONFIG['evaluation_status'])

## 1. Verify saved models and evaluate all comparisons

The evaluator checks saved model hashes, the data manifest, split separation and the frozen configuration. The fixed logistic models provide a same-row reference rather than relying on rounded numbers copied from a slide.

Two naive references are also recorded: the final-refit prevalence as a constant probability, and always predicting profitable. The latter is a majority-class-style classification reference, not pure random guessing; its log loss is large when any nonprofitable cases occur.

In [ ]:
import pandas as pd
metrics, predictions, differences = rf.evaluate_test(RUN_DIR, CONFIG)
display(metrics[['model', 'variant', 'rows', 'accuracy_threshold_0_5', 'roc_auc',
                 'log_loss', 'brier', 'pr_auc_average_precision']])

## 2. Compare history variants and algorithms

For log loss and Brier score, lower is better. For ROC-AUC and average precision, higher is better. Accuracy uses the fixed 0.5 threshold for every fitted model.

The paired comparison uses the same test rows. `alternative_minus_reference_log_loss < 0` favors the alternative. Event-only bootstrap intervals are descriptive: they do not refit models or capture repeated-wallet dependence across events. Report event concentration and avoid treating these reused-test intervals as confirmatory p-values.

In [ ]:
display(differences)
comparison = metrics[metrics.model.isin(['logistic', 'random_forest'])].pivot(
    index='variant', columns='model', values=['roc_auc', 'log_loss', 'accuracy_threshold_0_5'])
display(comparison)
summary = json.loads((RUN_DIR / 'evaluation_complete.json').read_text())
display(summary)

## 3. Plot ROC, calibration and individual importance

ROC and calibration each show exactly four fitted-model curves: logistic baseline, logistic with five prior-24h history features, random forest baseline, and random forest with the same history features. The precision–recall panel and smaller-feature detail figure are omitted.

The main importance comparison uses absolute standardized logistic coefficients and RF Gini scores from the final saved models. Grouped outer-validation permutation remains a supporting check. The HTML summary includes the RF tuning search, selected settings and CV selection method.

In [ ]:
figure_dir = rf.make_figures(RUN_DIR)
display(Image(filename=str(figure_dir / 'roc_calibration_four_models.png')))
display(Image(filename=str(figure_dir / 'variable_importance_comparison.png')))
display(Image(filename=str(figure_dir / 'validation_feature_importance.png')))
print('Summary HTML:', figure_dir / 'RF_logistic_resultssummary.html')

## 4. Read results without overstating them

Answer separately:

- Does RF improve prediction relative to the same-feature logistic reference?
- Does adding history improve RF relative to RF baseline?
- Which predictors have the largest standardized logistic coefficients or RF Gini scores?
- Does the supporting grouped validation permutation check agree about overall price/history reliance?

Importance is model-specific and is not causation or a significance test. A history feature can receive impurity importance without improving held-forward prediction. If gains are small or unstable, report that uncertainty and obtain a longer or future period instead of searching repeatedly for a favorable result.

The target remains conditional gross profitability. No output identifies confirmed insiders or verified net-profit opportunities.

In [ ]:
print('Outputs:', RUN_DIR)
print('Importance tables: variable_importance_all_models.csv, logistic_coefficient_importance.csv,')
print('random_forest_gini_importance.csv; supporting: validation_grouped_permutation_importance.csv')
print('Importance report and plots:', RUN_DIR / 'figures')
print('Other main tables: exploratory_test_metrics.csv, paired_event_log_loss_differences.csv, forward_cv_audit.csv')
print('Keep model_manifest.json and data_manifest.json with the results for reproducibility.')